<a href="https://colab.research.google.com/github/AshokGit544/ADF-Lakehouse-Ingestion-Project/blob/main/ADF_Lakehouse_Ingestion_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import json
import pandas as pd
import numpy as np

BASE_PATH = Path('/content/azure_data_factory_lakehouse_ingestion_demo')
SOURCE_PATH = BASE_PATH / 'data' / 'source'
LANDING_PATH = BASE_PATH / 'data' / 'landing'
OUTPUT_PATH = BASE_PATH / 'outputs'
ADF_PATH = BASE_PATH / 'adf'

for p in [SOURCE_PATH, LANDING_PATH, OUTPUT_PATH, ADF_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

num_customers = 1000
num_billing = 3000
num_meter = 12000
num_outages = 900
regions = ['North', 'South', 'East', 'West', 'Central']
start_date = datetime(2024, 1, 1)

print("Setup completed")

Setup completed


In [6]:
# customer master
customers = []
for i in range(1, num_customers + 1):
    customers.append({
        'customer_id': f'CUST{i:05d}',
        'customer_name': f'Customer_{i}',
        'region': random.choice(regions),
        'customer_type': random.choice(['Residential', 'Commercial', 'Industrial']),
        'source_system': 'CustomerSystem'
    })

pd.DataFrame(customers).to_csv(SOURCE_PATH / 'customers.csv', index=False)

# billing / sap fico style data
billing = []
for i in range(1, num_billing + 1):
    cust = random.randint(1, num_customers)
    bill_date = start_date + timedelta(days=random.randint(0, 179))
    billing.append({
        'billing_id': f'BILL{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'bill_date': bill_date.strftime('%Y-%m-%d'),
        'bill_amount': round(float(max(40, np.random.normal(145, 40))), 2),
        'sap_fico_doc_id': f'SAPFICO{i:08d}',
        'source_system': 'SAPFICO'
    })

pd.DataFrame(billing).to_csv(SOURCE_PATH / 'billing.csv', index=False)

# smart meter data
meter = []
for i in range(1, num_meter + 1):
    cust = random.randint(1, num_customers)
    read_date = start_date + timedelta(days=random.randint(0, 179))
    meter.append({
        'meter_record_id': f'MTRREC{i:07d}',
        'customer_id': f'CUST{cust:05d}',
        'reading_date': read_date.strftime('%Y-%m-%d'),
        'kwh_usage': round(float(max(5, np.random.normal(35, 10))), 2),
        'source_system': 'SmartMeterSystem'
    })

pd.DataFrame(meter).to_csv(SOURCE_PATH / 'meter_usage.csv', index=False)

# outage data
outages = []
for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    outage_date = start_date + timedelta(days=random.randint(0, 179))
    outages.append({
        'outage_id': f'OUT{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'outage_date': outage_date.strftime('%Y-%m-%d'),
        'duration_minutes': max(5, int(np.random.normal(80, 25))),
        'source_system': 'OutageSystem'
    })

pd.DataFrame(outages).to_csv(SOURCE_PATH / 'outages.csv', index=False)

print('Source datasets created successfully.')
for file in SOURCE_PATH.glob('*.csv'):
    print(file.name)

Source datasets created successfully.
outages.csv
meter_usage.csv
billing.csv
customers.csv


In [7]:
dataset_mapping = {
    'datasets': [
        {'source_file': 'customers.csv', 'target_file': 'customers_landing.csv', 'source_system': 'CustomerSystem'},
        {'source_file': 'billing.csv', 'target_file': 'billing_landing.csv', 'source_system': 'SAPFICO'},
        {'source_file': 'meter_usage.csv', 'target_file': 'meter_usage_landing.csv', 'source_system': 'SmartMeterSystem'},
        {'source_file': 'outages.csv', 'target_file': 'outages_landing.csv', 'source_system': 'OutageSystem'}
    ]
}

with open(ADF_PATH / 'dataset_mapping.json', 'w') as f:
    json.dump(dataset_mapping, f, indent=2)

print('Dataset mapping file created.')
print(json.dumps(dataset_mapping, indent=2))

Dataset mapping file created.
{
  "datasets": [
    {
      "source_file": "customers.csv",
      "target_file": "customers_landing.csv",
      "source_system": "CustomerSystem"
    },
    {
      "source_file": "billing.csv",
      "target_file": "billing_landing.csv",
      "source_system": "SAPFICO"
    },
    {
      "source_file": "meter_usage.csv",
      "target_file": "meter_usage_landing.csv",
      "source_system": "SmartMeterSystem"
    },
    {
      "source_file": "outages.csv",
      "target_file": "outages_landing.csv",
      "source_system": "OutageSystem"
    }
  ]
}


In [8]:
pipeline_definition = {
    'name': 'enterprise_lakehouse_ingestion_pipeline',
    'activities': [
        {'name': 'CopyCustomers', 'type': 'Copy', 'source': 'customers.csv', 'target': 'customers_landing.csv'},
        {'name': 'CopyBilling', 'type': 'Copy', 'source': 'billing.csv', 'target': 'billing_landing.csv'},
        {'name': 'CopyMeterUsage', 'type': 'Copy', 'source': 'meter_usage.csv', 'target': 'meter_usage_landing.csv'},
        {'name': 'CopyOutages', 'type': 'Copy', 'source': 'outages.csv', 'target': 'outages_landing.csv'}
    ],
    'description': 'ADF-style ingestion pipeline for enterprise source integration and Lakehouse landing'
}

with open(ADF_PATH / 'pipeline_definition.json', 'w') as f:
    json.dump(pipeline_definition, f, indent=2)

print('ADF-style pipeline definition created.')
print(json.dumps(pipeline_definition, indent=2))

ADF-style pipeline definition created.
{
  "name": "enterprise_lakehouse_ingestion_pipeline",
  "activities": [
    {
      "name": "CopyCustomers",
      "type": "Copy",
      "source": "customers.csv",
      "target": "customers_landing.csv"
    },
    {
      "name": "CopyBilling",
      "type": "Copy",
      "source": "billing.csv",
      "target": "billing_landing.csv"
    },
    {
      "name": "CopyMeterUsage",
      "type": "Copy",
      "source": "meter_usage.csv",
      "target": "meter_usage_landing.csv"
    },
    {
      "name": "CopyOutages",
      "type": "Copy",
      "source": "outages.csv",
      "target": "outages_landing.csv"
    }
  ],
  "description": "ADF-style ingestion pipeline for enterprise source integration and Lakehouse landing"
}


In [9]:
ingestion_logs = []

def log_ingestion(source_file, target_file, status, row_count, message=''):
    ingestion_logs.append({
        'source_file': source_file,
        'target_file': target_file,
        'status': status,
        'row_count': row_count,
        'timestamp': datetime.now().isoformat(),
        'message': message
    })

In [10]:
with open(ADF_PATH / 'dataset_mapping.json', 'r') as f:
    mapping = json.load(f)

for dataset in mapping['datasets']:
    source_file = SOURCE_PATH / dataset['source_file']
    target_file = LANDING_PATH / dataset['target_file']

    try:
        df = pd.read_csv(source_file)
        df['landing_timestamp'] = datetime.now().isoformat()
        df.to_csv(target_file, index=False)
        log_ingestion(dataset['source_file'], dataset['target_file'], 'SUCCESS', len(df), 'Copied successfully')
        print(f"Copied {dataset['source_file']} -> {dataset['target_file']}")
    except Exception as e:
        log_ingestion(dataset['source_file'], dataset['target_file'], 'FAILED', 0, str(e))
        print(f"Failed {dataset['source_file']}: {e}")

Copied customers.csv -> customers_landing.csv
Copied billing.csv -> billing_landing.csv
Copied meter_usage.csv -> meter_usage_landing.csv
Copied outages.csv -> outages_landing.csv


In [11]:
ingestion_log_df = pd.DataFrame(ingestion_logs)
ingestion_log_df.to_csv(OUTPUT_PATH / 'ingestion_log.csv', index=False)
ingestion_log_df

,source_file,target_file,status,row_count,timestamp,message
0,customers.csv,customers_landing.csv,SUCCESS,1000,2026-03-17T04:21:50.973907,Copied successfully
1,billing.csv,billing_landing.csv,SUCCESS,3000,2026-03-17T04:21:51.073246,Copied successfully
2,meter_usage.csv,meter_usage_landing.csv,SUCCESS,12000,2026-03-17T04:21:51.162650,Copied successfully
3,outages.csv,outages_landing.csv,SUCCESS,900,2026-03-17T04:21:51.169858,Copied successfully


In [12]:
summary = {
    'generated_at': datetime.now().isoformat(),
    'source_files': len(list(SOURCE_PATH.glob('*.csv'))),
    'landing_files': len(list(LANDING_PATH.glob('*.csv'))),
    'successful_copies': int((ingestion_log_df['status'] == 'SUCCESS').sum()),
    'failed_copies': int((ingestion_log_df['status'] == 'FAILED').sum())
}

with open(OUTPUT_PATH / 'ingestion_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "generated_at": "2026-03-17T04:23:09.106259",
  "source_files": 4,
  "landing_files": 4,
  "successful_copies": 4,
  "failed_copies": 0
}


In [13]:
print('Landing files created:')
for file in LANDING_PATH.glob('*.csv'):
    print(file.name)

pd.read_csv(LANDING_PATH / 'billing_landing.csv').head()

Landing files created:
billing_landing.csv
customers_landing.csv
outages_landing.csv
meter_usage_landing.csv


,billing_id,customer_id,bill_date,bill_amount,sap_fico_doc_id,source_system,landing_timestamp
0,BILL000001,CUST00510,2024-04-04,164.87,SAPFICO00000001,SAPFICO,2026-03-17T04:21:51.003310
1,BILL000002,CUST00947,2024-05-12,139.47,SAPFICO00000002,SAPFICO,2026-03-17T04:21:51.003310
2,BILL000003,CUST00274,2024-01-22,170.91,SAPFICO00000003,SAPFICO,2026-03-17T04:21:51.003310
3,BILL000004,CUST00745,2024-04-18,205.92,SAPFICO00000004,SAPFICO,2026-03-17T04:21:51.003310
4,BILL000005,CUST00081,2024-04-20,135.63,SAPFICO00000005,SAPFICO,2026-03-17T04:21:51.003310
